## Concept 2 — ChatPromptTemplate & Message Types

### What is it?
Chat models work with a list of typed messages — each tagged with a role.
`ChatPromptTemplate` lets you define this structure as a reusable, variable-driven template.

### Message Roles
```
system  → Persona, rules, constraints. Set once, always active.
human   → What the user says. Drives the conversation.
ai      → LLM's past response (used for multi-turn memory).
tool    → Output returned from a tool call (used in agent workflows).
```

### Pipeline
```
[SystemMessage, HumanMessage] → LLM → AIMessage → Answer
```

### Limitation (what Concept 3 fixes)
Instructions tell the LLM what to do — but no examples show what 'good' looks like.
→ Concept 3 adds Few-Shot examples so the LLM learns from demonstrations.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm, TEST_INPUTS, run_inputs
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
print('LLM ready')

### Step 1 — Message Types
Construct messages manually to understand each role before using templates.

In [ ]:
# Build a message list manually
messages = [
    SystemMessage(content='You are a concise Python expert. Answer in 1-2 sentences.'),
    HumanMessage(content='What is a decorator?'),
]

response = llm.invoke(messages)
print(f'Type: {type(response).__name__}')
print(f'Role: {response.response_metadata.get("finish_reason", "ai")}')
print(f'Content: {response.content}')

### Step 2 — ChatPromptTemplate
Define the message structure as a template with `{variables}` filled at runtime.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a {role}. Answer in {style}.'),
    ('human',  '{question}'),
])

# Inspect the formatted message list
messages = prompt.format_messages(
    role='Python expert',
    style='bullet points',
    question='What are Python data types?'
)

for msg in messages:
    print(f'[{msg.type}] {msg.content}')

### Step 3 — How the System Role Controls Output
The same human question gives very different answers depending on the system message.

In [ ]:
question = 'What is recursion?'

personas = [
    ('concise expert',         'one sentence'),
    ('teacher for beginners',  'a simple analogy'),
    ('strict code reviewer',   'a code example only'),
]

for role, style in personas:
    msgs = prompt.format_messages(role=role, style=style, question=question)
    resp = llm.invoke(msgs)
    print(f'\n[{role}]\n{resp.content}')
    print('-' * 40)

### Step 4 — LCEL Chain with ChatPromptTemplate

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful technical assistant. Be clear and concise.'),
    ('human',  '{input}'),
])

chain = chat_prompt | llm | StrOutputParser()

result = chain.invoke({'input': 'What is polymorphism?'})
print(result)

### Full Function

In [ ]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful technical assistant. Answer clearly and concisely.'),
    ('human',  '{input}'),
])
final_chain = final_prompt | llm | StrOutputParser()

def chat_prompt_rag(query):
    return final_chain.invoke({'input': query})

### Test — Standard Inputs

In [ ]:
run_inputs(chat_prompt_rag, label='Concept 2 — ChatPromptTemplate')